# Scraper Showdown: BeautifulSoup vs Scrapy vs Crawl4AI

A practical comparison for **AI-driven workflows**. We'll scrape a JavaScript-rendered page and compare:

- **Lines of Code** required
- **Ease of Use** for developers
- **AI-Readiness** of the output

**Target:** `https://quotes.toscrape.com/js/` — quotes rendered entirely via JavaScript.

**Setup:** `pip install beautifulsoup4 requests playwright scrapy crawl4ai pandas`
**Then:** `playwright install chromium`


In [5]:
import asyncio
import time
import pandas as pd
import nest_asyncio

# Allow asyncio to work in Jupyter notebooks
nest_asyncio.apply()

TARGET_URL = "https://quotes.toscrape.com/js/"

## 1. BeautifulSoup: Manual Labor & External Dependencies

BeautifulSoup **cannot execute JavaScript**. We need:

1. Playwright for JS rendering
2. Manual HTML parsing
3. Manual text cleaning

**The traditional approach — powerful but labor-intensive.**


In [6]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


async def scrape_with_beautifulsoup(url):
    start = time.time()

    # Step 1: Render JavaScript (BS4 can't do this)
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, wait_until="networkidle")
        html = await page.content()
        await browser.close()

    # Step 2: Manual parsing & cleaning
    soup = BeautifulSoup(html, "html.parser")
    quotes = []

    for block in soup.select("div.quote"):
        text = block.select_one("span.text").get_text(strip=True).strip("'").strip('")')
        author = block.select_one("small.author").get_text(strip=True)
        quotes.append({"quote": text, "author": author})

    # Step 3: Manual markdown conversion
    markdown = "\n".join([f"> {q['quote']}\n> — *{q['author']}*" for q in quotes])

    return {"quotes": quotes, "markdown": markdown, "time": time.time() - start}


# Test BeautifulSoup
bs_result = await scrape_with_beautifulsoup(TARGET_URL)
print(f"BeautifulSoup: {len(bs_result['quotes'])} quotes in {bs_result['time']:.2f}s")
print(f"Sample:\n{bs_result['markdown'][:200]}...")

BeautifulSoup: 10 quotes in 1.73s
Sample:
> “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
> — *Albert Einstein*
> “It is our choices, Harry, that show what we truly are, fa...


## 2. Scrapy: Industrial Framework with Boilerplate

Scrapy is built for **large-scale crawling**. Even for one page, you need:

- Spider class with `parse()` method
- CrawlerProcess configuration
- Global state handling

**Note:** Standard Scrapy can't render JS. Requires `scrapy-playwright` middleware.


In [7]:
import scrapy
from scrapy.crawler import CrawlerProcess

# Global container (Scrapy pattern)
scrapy_quotes = []


class QuotesSpider(scrapy.Spider):
    name = "quotes"
    start_urls = [TARGET_URL]
    custom_settings = {"LOG_LEVEL": "ERROR"}

    def parse(self, response):
        for block in response.css("div.quote"):
            text = block.css("span.text::text").get("").strip("'").strip('")')
            author = block.css("small.author::text").get("")
            if text and author:
                scrapy_quotes.append({"quote": text, "author": author})


print(f"\nScrapy results: {len(scrapy_quotes)} quotes (would be 0 without scrapy-playwright)")


Scrapy results: 0 quotes (would be 0 without scrapy-playwright)


## 3. Crawl4AI: One-Line Async Extraction for AI

Crawl4AI is **purpose-built for AI workflows**:

- JavaScript rendering (built-in)
- Automatic markdown conversion
- LLM-optimized output
- Native async support

**Zero boilerplate, zero configuration.**


In [8]:
from crawl4ai import AsyncWebCrawler


async def scrape_with_crawl4ai(url):
    start = time.time()
    async with AsyncWebCrawler(verbose=False) as crawler:
        result = await crawler.arun(url=url)
    return {"markdown": result.markdown, "metadata": result.metadata, "time": time.time() - start}


# Run Crawl4AI
c4ai_result = await scrape_with_crawl4ai(TARGET_URL)
print(f"Crawl4AI: {len(c4ai_result['markdown'])} chars of markdown in {c4ai_result['time']:.2f}s")
print(f"\nLLM-Ready Output:\n{c4ai_result['markdown'][:300]}...")

[INIT].... → Crawl4AI 0.7.8 

[FETCH]... ↓ https://quotes.toscrape.com/js/                                                                      |
✓ | ⏱: 1.11s 

[SCRAPE].. ◆ https://quotes.toscrape.com/js/                                                                      |
✓ | ⏱: 0.00s 

[COMPLETE] ● https://quotes.toscrape.com/js/                                                                      |
✓ | ⏱: 1.11s 

Crawl4AI: 1666 chars of markdown in 2.14s

LLM-Ready Output:
#  [Quotes to Scrape](https://quotes.toscrape.com/)
[Login](https://quotes.toscrape.com/login)
“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”by Albert Einstein
Tags: change deep-thoughts thinking world
“It is our choices, Harry, th...
